# Lecture: Attention — Letting Words Look at Each Other (QKV, Masks, Heads)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain queries, keys, and values as "question, label, content" in one sentence each (§4).
- Explain why we divide scores by √d and what a causal mask blocks (§5–§6).
- Read the system map (§9) and narrate one head: tokens → QKV → mask → softmax → mix.


## The big idea

Imagine each word in a sentence holding up a question card ("what am I looking for?" — the **query**), a label card ("what do I offer?" — the **key**), and a content card ("what do I actually say?" — the **value**). Every word compares its question against all labels (a dot product = similarity score), turns the scores into vote shares with **softmax** (all positive, summing to 1), and collects a weighted mix of everyone's content cards — those vote shares are the **attention weights**. Without rescaling, big dot products make softmax output a winner-take-all 1.0 that kills learning, so we divide by **√d (scaling)** to keep votes soft. When generating text left-to-right, a **causal mask** covers future words' cards so each word votes on the past only. Real models run several such votes in parallel (**multi-head attention**, fused in torch as **SDPA**) and add an order signal (**positional encoding**) because attention itself is blind to word order.


In [ ]:
import numpy as np  # bring in numpy for score math and arrays
import matplotlib.pyplot as plt  # bring in pyplot for the heatmap + bars
np.random.seed(10)  # fix the seed so scores are identical every run
words = ['cat', 'sat', 'on', 'mat']  # tiny 4-token toy sentence
n = 4  # number of tokens in the toy sentence
raw = np.array([[4., 1., 0., 0.],  # row 0: 'cat' looks mostly at itself
                [2., 3., 1., 0.],  # row 1: 'sat' splits cat/self
                [1., 1., 3., 2.],  # row 2: 'on' spreads wider
                [0., 1., 2., 4.]])  # row 3: 'mat' looks back at 'on'/self
m = raw.max(axis=1, keepdims=True)  # row max for numerical stability
w = np.exp(raw - m) / np.exp(raw - m).sum(axis=1, keepdims=True)  # softmax rows into votes
print('row sums:', np.round(w.sum(axis=1), 6))  # every row must sum to 1.0
print("row for 'sat':", np.round(w[1], 3))  # one full vote distribution: mostly cat + self
fig, ax = plt.subplots(1, 2, figsize=(9, 3))  # make heatmap + bar panels
im = ax[0].imshow(w, cmap='Blues', vmin=0, vmax=1)  # draw votes as a heatmap
ax[0].set_xticks(range(n))  # place one tick per column token
ax[0].set_xticklabels(words)  # label columns with words
ax[0].set_yticks(range(n))  # place one tick per row token
ax[0].set_yticklabels(words)  # label rows with words
ax[0].set_title('who looks at whom (rows sum to 1)')  # title the heatmap
fig.colorbar(im, ax=ax[0], shrink=0.8)  # add a vote-share colorbar
ax[1].bar(words, w[1], color='steelblue')  # bar chart of the 'sat' row votes
ax[1].set_ylim(0, 1)  # fix the bar range to valid vote shares
ax[1].set_title("'sat' mixes these contents")  # title the bar panel
fig.suptitle('Attention = similarity scores -> vote shares -> content mix')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 builds the vote, §5–§6 keep it trainable and honest, §7–§8 scale it up.

| Term | One-liner | Section |
|---|---|---|
| Query (Q) | Each token's "question card": what am I looking for in others? | §4 |
| Key (K) | Each token's "label card": what do I offer to others' questions? | §4 |
| Value (V) | Each token's "content card": what gets mixed in proportion to votes | §4 |
| Attention weights | The softmax vote shares (rows sum to 1) deciding each mix | §4 |
| Softmax | Score-to-votes function: exponentiate, then divide by the total | §4 |
| Scaling (1/√d) | Dividing scores by √head-size so votes stay soft and learnable | §5 |
| Causal mask | Cover over future tokens so generation can't peek ahead | §6 |
| Multi-head attention (MHA) | Several small attentions in parallel subspaces, then merged | §7 |
| SDPA | PyTorch's fast fused attention kernel (same math as the hand version) | §7 |
| Positional encoding | Added order signal (sinusoid curves or learned table): attention is order-blind | §8 |


### §4 Queries, keys, values, attention weights

**In one sentence: each token's query asks "what am I looking for", every token's key answers "what do I offer", their dot product is a similarity score, and softmax turns each row of scores into attention weights that mix everyone's values.**

Three learned matrices make it happen: `Q = X @ Wq`, `K = X @ Wk`, `V = X @ Wv` — same input `X`, three different projections, three different jobs. The scores `Q @ K.T` form an n×n table (row i = "who does token i look at"), and the mix `W @ V` collects the content. Watch it: 3 toy tokens of width 4 project to Q/K/V of width 2, and the resulting 3×3 vote table has rows summing to 1 — row 0 spreads [0.14, 0.58, 0.28], i.e. token 0 listens most to token 1, and the mix collects token 1's content most.


In [ ]:
import numpy as np  # reuse numpy for the QKV projections and votes
import matplotlib.pyplot as plt  # reuse pyplot for the vote heatmap
import math  # reuse math for the sqrt(dk) scale
np.random.seed(10)  # fix the seed for reproducibility
X = np.array([[1., 0., 1., 0.],  # token 0 embedding (width 4)
              [0., 1., 0., 1.],  # token 1 embedding
              [1., 1., 0., 0.]])  # token 2 embedding
Wq = np.array([[1., 0.], [0., 1.], [1., -1.], [0., 0.]])  # query projector 4->2
Wk = np.array([[1., 1.], [0., 1.], [-1., 0.], [1., 0.]])  # key projector 4->2
Wv = np.array([[2., 0.], [0., 2.], [1., 1.], [-1., 1.]])  # value projector 4->2 (the content)
Q = X @ Wq  # questions: (3, 2)
K = X @ Wk  # labels: (3, 2)
V = X @ Wv  # contents: (3, 2)
s = (Q @ K.T) / math.sqrt(2)  # scaled similarity scores (3, 3)
m = s.max(axis=1, keepdims=True)  # row max for stable softmax
w = np.exp(s - m) / np.exp(s - m).sum(axis=1, keepdims=True)  # attention weights: rows sum to 1
out = w @ V  # mix values by votes: (3, 2)
print('Q shape', Q.shape, 'K shape', K.shape, 'V shape', V.shape)  # all (3, 2): 3 tokens, width 2
print('row sums:', np.round(w.sum(axis=1), 6))  # the softmax contract: every row is 1.0
print('row 0 votes:', np.round(w[0], 3))  # token 0 listens most to token 1: [0.14, 0.58, 0.28]
print('output[0]:', np.round(out[0], 3))  # token 0's collected content vector
fig, ax = plt.subplots(figsize=(4, 3.5))  # make one panel for the vote table
im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)  # draw the 3x3 votes
ax.set_xticks(range(3))  # tick per key token
ax.set_xticklabels(['tok0', 'tok1', 'tok2'])  # label the keys
ax.set_yticks(range(3))  # tick per query token
ax.set_yticklabels(['tok0', 'tok1', 'tok2'])  # label the queries
ax.set_title('3-token vote table (rows sum to 1)')  # title the panel
fig.colorbar(im, shrink=0.8)  # add a vote-share colorbar
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §5 Scaling — why divide by √d

**In one sentence: dot products grow with head width (variance ≈ d), so without dividing by √d the scores explode, softmax collapses to a 1.0 one-hot, and gradients die — scaling restores soft, learnable votes.**

With head width 64, two random vectors' dot product scatters with standard deviation 8 — exponentiating gaps that wide picks a single winner every time. Dividing by √64 = 8 squeezes the spread back to ~1, where softmax hedges across candidates and gradients flow. Watch it: the same query against 4 keys gives a collapsed no-scale vote (max-p ≈ 1.0) versus a soft scaled vote (max-p ≈ 0.61) — the lab's experiment table shows the same collapse on its own data.


In [ ]:
import numpy as np  # reuse numpy for softmax and score demos
import matplotlib.pyplot as plt  # reuse pyplot for the side-by-side bars
np.random.seed(10)  # fix the seed so the demo numbers are identical every run
dk = 64  # large head size where missing scale hurts most
q = np.random.normal(size=dk)  # one random query vector of width 64
K = np.random.normal(size=(4, dk))  # four random key vectors to compare against
s = K @ q  # raw dot scores: variance grows with dk
print('raw scores:', np.round(s, 2))  # unscaled similarities: wide spread
m = s.max()  # max for stable softmax
w_noscale = np.exp(s - m) / np.exp(s - m).sum()  # softmax WITHOUT 1/sqrt(dk)
ss = s / np.sqrt(dk)  # scale first: spread restored to ~1
e = np.exp(ss - ss.max())  # stable exponentiate of scaled scores
w_scale = e / e.sum()  # normalize scaled votes to sum to 1
print('no-scale max-p:', round(float(w_noscale.max()), 4))  # ~1.0: collapsed winner-take-all vote
print('scaled max-p:', round(float(w_scale.max()), 4))  # ~0.61: soft spread, gradients flow
fig, ax = plt.subplots(1, 2, figsize=(9, 3))  # two bar panels: scale off vs on
ax[0].bar(['t0', 't1', 't2', 't3'], w_noscale, color='tomato')  # collapsed one-hot bars
ax[0].set_ylim(0, 1)  # fix the vote-share range
ax[0].set_title('no 1/sqrt(d): winner-take-all')  # title the left panel
ax[1].bar(['t0', 't1', 't2', 't3'], w_scale, color='steelblue')  # soft spread bars
ax[1].set_ylim(0, 1)  # fix the vote-share range
ax[1].set_title('with 1/sqrt(d): soft votes')  # title the right panel
fig.suptitle('Scaling keeps votes soft and trainable')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


### §6 Causal mask — no peeking at the future

**In one sentence: the causal mask adds −∞ (in practice −1e9) to every future slot before softmax, so token i mixes only tokens ≤ i — generation stays honestly left-to-right.**

The mask is just a lower triangle of allowed slots (diagonal included: each token may see itself). Softmax turns −1e9 into a weight of exactly 0.0, and the leak check proves it: the biggest weight on any future slot is 0.0. Watch it: the mask matrix plus the masked first row — token 0's votes are `[1, 0, 0, 0]` (alone with itself), while the tempting future score of 9.0 in its row contributes nothing.


In [ ]:
import numpy as np  # reuse numpy for masks and masked softmax
import matplotlib.pyplot as plt  # reuse pyplot for the mask picture
np.random.seed(10)  # fix the seed for reproducibility
mask = np.tril(np.ones((4, 4), dtype=bool))  # lower triangle: past + self allowed, future blocked
smat = np.array([[2., 3., 9., 9.],  # toy scores: 9s are tempting future peaks
                 [1., 2., 8., 8.],  # future columns must be masked out
                 [1., 1., 2., 7.],  # row 2 may use columns 0..2 only
                 [0., 1., 1., 2.]])  # last row sees everything
masked = np.where(mask, smat, -1e9)  # future slots get -inf so softmax gives ~0
mm = masked.max(axis=1, keepdims=True)  # row max for stable softmax
wm = np.exp(masked - mm) / np.exp(masked - mm).sum(axis=1, keepdims=True)  # masked votes
print('masked row 0:', np.round(wm[0], 3))  # expect [1, 0, 0, 0]: the 9s vanish
print('future leak (max weight on any future slot):', round(float(wm[~np.tril(np.ones((4, 4), dtype=bool))].max()), 6))  # must be 0.0
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))  # two panels: the mask, then its effect
ax[0].imshow(mask, cmap='Greens', vmin=0, vmax=1)  # draw allowed (green) vs blocked (white)
ax[0].set_title('causal mask: green = may look')  # title the left panel
ax[1].bar(['t0', 't1', 't2', 't3'], wm[0], color='steelblue')  # token 0's votes after masking
ax[1].set_ylim(0, 1)  # fix the vote-share range
ax[1].set_title("token 0 votes: future 9s give 0")  # title the right panel
fig.suptitle('Masking blocks the future (leak = 0)')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


### §7 MHA and SDPA — many small attentions at once

**In one sentence: multi-head attention splits the width d into h parallel subspaces that attend independently and merge through one output matrix, and SDPA is PyTorch's fast fused kernel computing exactly that same math.**

Splitting is just reshaping: with d = 32 and h = 4 heads, each head works in width 8 (`dh = d / h`), scores its own n×n table with §5's scale and §6's mask, and the four 8-wide outputs concatenate back to 32 before the final mix. The parameter count is beautifully simple — four square d×d matrices (Q, K, V, output) = 4 × 32 × 32 = 4096 numbers. Watch it: the diagram traces one token batch forking into 4 heads and rejoining, with the 4096-param receipt printed below.


In [ ]:
import numpy as np  # reuse numpy for the parameter arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the heads diagram
np.random.seed(10)  # fix the seed for reproducibility
d, h = 32, 4  # toy width and head count (lab MHA defaults)
dh = d // h  # per-head width: 8
params = 4 * d * d  # Q, K, V, output: four square matrices, no biases
print('per-head width dh =', dh)  # expect 8: each head's private subspace
print('MHA params = 4 x 32 x 32 =', params)  # expect 4096: attention is 4 square matrices
print('SDPA = same math, fused kernel (lab verifies hand-vs-torch to ~1e-12)')  # the speed story
fig, ax = plt.subplots(figsize=(9, 3.5))  # make one wide panel for the fork diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 12)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
ax.text(1.5, 2, 'tokens\n(n x 32)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # input box
for k in range(4):  # draw one box per attention head
    ax.text(4.5 + k * 1.2, 2.6, f'head{k}\n(n x 8)', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # head box
ax.text(10, 2, 'concat + Wo\n(n x 32)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # merge box
ax.text(6.3, 1.0, 'each head: scores / √8 -> causal mask -> softmax -> xV (§5, §6)', ha='center', fontsize=8, color='darkred')  # per-head recipe
ax.text(6, 3.4, 'MHA: split width 32 into 4 x 8, attend in parallel, rejoin (4096 params)', ha='center', fontsize=9, fontweight='bold')  # banner
for k in range(4):  # fan-out arrows from input to heads
    ax.annotate('', xy=(4.5 + k * 1.2, 2.9), xytext=(2.3, 2.2), arrowprops=dict(arrowstyle='->', color='gray'))  # fork arrow
ax.annotate('', xy=(9.3, 2), xytext=(8.3, 2), arrowprops=dict(arrowstyle='->'))  # heads to merge arrow
ax.set_title('Multi-head attention: divide, attend, conquer')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §8 Positions — telling the model about order

**In one sentence: attention is order-blind (shuffling tokens shuffles outputs identically), so positional encodings add an order signal — fixed sinusoid curves or a learned position table — letting the model tell "cat sat" from "sat cat".**

Sinusoids are the elegant option: dimension pairs oscillate at geometric wavelengths (from 2π to 2π×10000), so nearby positions get similar vectors and far ones differ — self-similarity 4.0 (width 8 = four sin²+cos² pairs summing to 1 each) versus 3.52 ten positions away. The lab's learned table does the same job with parameters instead of math. Watch it: four wavy dims across 16 positions, with self-similarity 4.0 (the ceiling — identical rows always win) versus 3.52 ten positions away printed.


In [ ]:
import numpy as np  # reuse numpy for the sinusoid table
import matplotlib.pyplot as plt  # reuse pyplot for the wave plot
import math  # reuse math for log in the wavelength ladder
np.random.seed(10)  # fix the seed for reproducibility
n, d = 16, 8  # 16 positions, width 8 (4 sin/cos pairs)
pe = np.zeros((n, d))  # the position table
pos = np.arange(n)[:, None]  # (16, 1): token positions
div = np.exp(np.arange(0, d, 2) * -(math.log(10000.0) / d))  # geometric wavelengths
pe[:, 0::2] = np.sin(pos * div)  # sin on even dims
pe[:, 1::2] = np.cos(pos * div)  # cos on odd dims
self_sim = float(pe[3] @ pe[3])  # position 3 vs itself
dist_sim = float(pe[3] @ pe[10])  # position 3 vs far position 10
print('self-similarity:', round(self_sim, 2))  # expect 4.0: four pairs each summing to 1
print('distant similarity:', round(dist_sim, 2))  # expect ~3.5 (still high: toy d=8 barely decorrelates at distance 7; try distance 15 or larger d to see decay)
fig, ax = plt.subplots(figsize=(7, 4))  # make one panel for the waves
for j in range(4):  # draw the first four dims
    ax.plot(range(n), pe[:, j], marker='o', ms=3, label=f'dim {j}')  # wave for this dim
ax.set_xlabel('position')  # label the order axis
ax.set_ylabel('encoding value')  # label the signal axis
ax.set_title('Sinusoid positions: each dim hums at its own wavelength')  # title the story
ax.legend(fontsize=8)  # identify the dims
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map (§9)

One attention head is a pipeline where every § term holds one station. Tokens enter as embeddings, fork into **Q/K/V** (§4), and form the score table. Scores are **scaled** by √d (§5) so votes stay soft, **masked** to the past (§6) so generation stays honest, softmaxted into **weights**, and mixed with V. In a full layer that dance runs h times in parallel (**MHA/SDPA**, §7) over position-tagged inputs (§8). The trace below replays the core on 3 tokens with a causal mask: row 0 can only see token 0, so its votes are exactly `[1, 0, 0]` — the mask doing its §6 duty inside the §9 line.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(10)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('tokens\n+Xpos §8', 1), ('QKV §4\nproject', 3.6), ('scores/√d §5\n+mask §6', 6.4), ('softmax\nweights', 9), ('xV -> out\nMHA §7', 11.8)]  # pipeline stations
colors = ['wheat', 'lightblue', 'plum', 'lightgreen', 'steelblue']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(6.6, 3.5, 'ONE HEAD: project (§4) -> scale (§5) -> mask (§6) -> vote -> mix (x h heads §7)', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(1.9, 2.8), (4.5, 5.4), (7.4, 8.3), (9.7, 11.0)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.set_title('System map: one attention head through every vocabulary term')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the end-to-end numeric trace
import math  # reuse math for sqrt(dk)
np.random.seed(10)  # fix the seed (arrays below are hardcoded; habit kept)
s = np.array([[4., 1., 0.],  # row 0: token 0's raw scores (§4)
              [2., 3., 1.],  # row 1: token 1 looks back at 0 and self
              [1., 1., 3.]])  # row 2: token 2 spreads
dk = 2  # toy head width: scale is sqrt(2)
mask = np.tril(np.ones((3, 3), dtype=bool))  # causal cover (§6)
sc = np.where(mask, s / math.sqrt(dk), -1e9)  # scale (§5), then mask (§6)
m = sc.max(axis=1, keepdims=True)  # row max for stable softmax
w = np.exp(sc - m) / np.exp(sc - m).sum(axis=1, keepdims=True)  # votes (§4)
print('trace weights =')  # label the vote table
print(np.round(w, 3))  # row 0 must read [1, 0, 0]: masked future contributes nothing
print('row sums:', np.round(w.sum(axis=1), 6))  # the contract holds: all 1.0
print('row 0:', np.round(w[0], 3), '-> token 0 sees only itself (§6)')  # the headline check


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, with proofs attached:

1. **Cell 1 (setup)** — seeds, device pick, `os.makedirs('./results')`. Same `np.random.seed` habit as this lecture.
2. **Cell 2 (single-head attention)** — `attn_numpy` (Q = XWq, K = XWk, V = XWv, scores / √dk, softmax, mix) verified against torch's fused SDPA to ~1e-12, `causal_mask` unit-tested for zero future leak (§4–§6, your trace scaled up), plus a median-of-reps CPU/GPU bench.
3. **Cell 3 (multi-head + positions)** — the `MHA` module splits width d into h subspaces that attend in parallel then merge (§7: 4 heads × d = 32 = 4096 params), with sinusoidal vs learned positional encodings adding order (§8) because attention itself is permutation-invariant.
4. **Cell 4 (experiments)** — no-scale max-p ≈ 1.0 vs scaled ≈ 0.61 (your §5 bars), causal-vs-full output diff > 0 (future leaks without the mask), and timing over lengths 128–1024 with a log-log slope ≈ 2.0 proving the O(n²) score matrix — the bottleneck P12's KV-cache will attack.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **What are Q, K, V in one sentence each, and what shape is each for 3 tokens of width 4 with head width 2?**

*Answer: Q is what each token looks for, K is what each token offers to be matched against, V is the content mixed by the votes (§4) — all three are (3, 2) in the §4 cell: 3 tokens, head width 2.*

2. **Why divide by √d before softmax, and what number proves it matters?**

*Answer: raw dot products grow with width and collapse softmax to a 1.0 one-hot with dead gradients (§5) — the §5 cell shows no-scale max-p ≈ 1.0 versus scaled ≈ 0.61 on the same query.*

3. **What does the causal mask do, and how do you prove it leaks nothing?**

*Answer: it writes −1e9 into future slots so each row mixes only the past (§6) — the leak check takes the max weight over all future slots and gets exactly 0.0, and the §9 trace's row 0 reads `[1, 0, 0]`.*
